In [ ]:
###--- Load libraries and set the location path for analysis data ---###

In [ ]:
import numpy as np
import scanpy as sc
import pandas as pd
import scipy.io
import matplotlib as mpl
import batchglm.api as glm
import diffxpy.api as de
import decoupler as dc

from matplotlib import rcParams
import bbknn
import os
import sys
import scipy
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scrublet as scr

In [ ]:
sc.settings.verbosity = 2  # show logging output
sc.settings.dir = "scRNA_out/"
sc.settings.autosave = True  # save figures, do not show them
sc.settings.figdir = "scRNA_out/figure"
sc.settings.set_figure_params(dpi=200, format="pdf", dpi_save=400)

In [ ]:
###--- Load pre-filtering data ---###

In [ ]:
# Data Loading
adata = sc.read("sc_clustering_global_harmony.h5ad")
adata.X = adata.layers["counts"].copy()

In [ ]:
###--- IDENTIFYING CELLULAR STRUCTURE ---###

In [ ]:
# Start with Raw data
adata.X = adata.layers["counts"].copy()
del (adata.uns['log1p'])
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
adata.raw = adata
adata.raw.var.index.is_unique
adata.raw.X

In [ ]:
#- Differentially expressed genes: TOP MARKER -#
sc.tl.rank_genes_groups(adata, groupby='leiden', reference='rest', method='wilcoxon' , key_added="dea_leiden", pts=True)

top_markers = pd.DataFrame(adata.uns['dea_leiden']['names']).head(100)
print(top_markers)

In [ ]:
# save results
result = adata.uns['dea_leiden']
groups = result['names'].dtype.names
result_df = pd.DataFrame(
    {group + '_' + key[:10]: result[key][group]
    for group in groups for key in ['names', 'scores', 'logfoldchanges', 'pvals','pvals_adj','pts','pts_rest']}).head(100)
result_df
result_df.to_csv("differential_expression_results_All_clusters.csv", index=False)

In [ ]:
# From markers to cluster cell_type annotation refine
cell_name = {'0': 'Macrophages',
'1': 'B cells',
'2': 'B cells',            
'3': 'B cells',
'4': 'B cells',
'10': 'B cells',
'5': 'Prolif B',
'6': 'Mast cells',   
'7': 'Neutrophils',
'8': 'NK cells',  
'9': 'Pre-B',
'11': 'Canonical DCs', 
'14': 'Plasmacytoid DCs',           
'12': 'Cancer Cell',           
'13': 'T cells',              
'15': 'T cells',  
'16': 'Unknown',             
'17': 'Plasma cells'
}

adata.obs["cell_type"] = adata.obs.leiden.map(cell_name)

In [ ]:
# cell_type counts
cell_type_counts = adata.obs['cell_type'].value_counts()
print(cell_type_counts)

In [ ]:
# Set umap palette colors
umap_colors = {'B cells': '#279E68',
'Macrophages': '#3A99BA',
'Prolif B': '#D3D978',              
'Mast cells': '#F0B9C6',
'NK cells': '#D0B395',
'Neutrophils': '#83D4F2',            
'Pre-B': '#8C9D61',              
'T cells': '#C97563', 
'Unknown': '#B6B6B6',
'Cancer Cell': '#5A5A5A',
'Canonical DCs': '#DAB6D0',
'Plasmacytoid DCs': '#EAB58A',
'Plasma cells': '#FDCB45'
}

In [ ]:
sc.pl.umap(adata, color='cell_type', save='_Global_cell_specific_cell_type_legend', palette=umap_colors)

In [ ]:
# Specific family Marker genes
marker_genes_dict = {
'Cancer Cells': ['EPCAM', 'EGFR', 'MET', 'CEACAM5'],    
'Proliferation': ['MKI67','TOP2A', 'STMN1'],
'Lymphoid marker gens': ['MS4A1', 'CD19', 'CD24', 'CD79A', 'CD34', 'JCHAIN', 'IGKC', 'MZB1', 'IGLL5', 'CD3E', 'CD8A',  'IL7R', 'IL2RA', 'NCAM1', 'KLRD1', 'KLRF1', 'KLRC1',  'NKG7', 'TRDC'],
'Myeloid marker gens': ['CD68', 'MRC1',  'LYZ', 'FCN1', 'CSF3R', 'ARG1', 'NRXN1','TLR2', 'CD1C', 'CLEC9A', 'ZBTB46','LILRA4', 'IL3RA', 'CLEC4C', 'TLR7', 'KIT', 'GATA2', 'CCR3','HPGDS']
}

In [ ]:
sc.pl.dotplot(adata, var_names=marker_genes_dict, groupby='cell_type', color_map='viridis', vmin=0, vmax=1, standard_scale='var', save='Global_marker_specificgenes_family_annotation_def');

In [ ]:
# Save the AnnData object
adata_ca = adata.copy()
adata_ca.write("sc_allcells_annotation_global.h5ad")
del adata_ca

In [ ]:
###--- Create the AnnData object lineage categories ---###

In [ ]:
# List of desired annotation categories
myeloid_categories = ['Macrophages', 'Mast cells', 'Neutrophils' ,'Canonical DCs','Plasmacytoid DCs']
lymphoid_categories = ['B cells','T cells','Prolif B','Plasma cells','NK cells','Pre-B']

# Create a boolean series indicating whether each annotation is in the list of desired categories
boolean_mask1 = adata.obs['cell_type'].isin(lymphoid_categories)
boolean_mask2 = adata.obs['cell_type'].isin(myeloid_categories)

#Create a new AnnData object including only the rows with the desired categories
subset_adata_ly = adata[boolean_mask1, :]
subset_adata_my = adata[boolean_mask2, :]

In [ ]:
# This simply save the state of the AnnData object Counts.
adata_my = subset_adata_my.copy()
adata_my.write("sc_celltypist_myeloid.h5ad")
del adata_my

In [ ]:
# This simply save the state of the AnnData object Counts.
adata_ly = subset_adata_ly.copy()
adata_ly.write("sc_celltypist_lymphoid.h5ad")
del adata_ly

In [ ]:
###--- Evaluation of cell typy distribution ---###

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# List of annotation categories
comparison_id= ['UT1','UT2','UT3','CAR1','CAR2','CAR3']

# Create a boolean series indicating whether each annotation is in the list of desired categories
boolean_mask = subset_adata_ly.obs['sample'].isin(comparison_id)
subset_adata_ly_cmp = subset_adata_ly[boolean_mask, :]

# Create the cross-tabulated data
tmp = pd.crosstab(subset_adata_ly_cmp.obs['sample'], subset_adata_ly_cmp.obs['cell_type'], normalize='index')
# Define the order of samples and cell types
sample_order = ['UT1', 'UT2', 'UT3', 'CAR1', 'CAR2', 'CAR3']
cell_type_order = ['Pre-B','Prolif B', 'B cells', 'T cells', 'NK cells','Plasma cells']
tmp = tmp.loc[sample_order, cell_type_order]

# Create the horizontal stacked bar plot
ax = tmp.plot(kind='bar',stacked=True, edgecolor='none', color=umap_colors)

# Define labels and legend
horiz_offset = 1.03
vert_offset = 1.
ax.legend(bbox_to_anchor=(horiz_offset, vert_offset))
ax.set_ylabel("Normalized Counts")
ax.set_xlabel("Sample")
ax.set_title("Normalized Stacked Bar Plot of Annotation Categories")

# Save the plot
plt.savefig('scRNA_out/figure/lymphoid_categories_barplot.pdf', bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Create the cross-tabulated data
tmp = pd.crosstab(subset_adata_ly_cmp.obs['label'], subset_adata_ly_cmp.obs['cell_type'], normalize='index')

# Create the horizontal stacked bar plot
sample_order = ['UT', 'CAR']
cell_type_order = ['Pre-B','Prolif B', 'B cells', 'T cells', 'NK cells','Plasma cells']
tmp = tmp.loc[sample_order, cell_type_order]
ax = tmp.plot(kind='bar',stacked=True, edgecolor='none', color=umap_colors)

# Define labels and legend
horiz_offset = 1.03
vert_offset = 1.
ax.legend(bbox_to_anchor=(horiz_offset, vert_offset))
ax.set_ylabel("Normalized Counts")
ax.set_xlabel("Sample")
ax.set_title("Normalized Stacked Bar Plot of Annotation Categories")

# Save the plot
plt.savefig('scRNA_out/figure/lymphoid_categories_barplot_design.pdf', bbox_inches='tight')

# Show the plot DEL
plt.show()

In [ ]:
# Create a boolean series indicating whether each annotation is in the list of desired categories
boolean_mask = subset_adata_my.obs['sample'].isin(comparison_id)
subset_adata_my_cmp = subset_adata_my[boolean_mask, :]

# Create the cross-tabulated data
tmp = pd.crosstab(subset_adata_my_cmp.obs['sample'], subset_adata_my_cmp.obs['cell_type'], normalize='index')
# Define the order of samples and cell types
sample_order = ['UT1', 'UT2', 'UT3', 'CAR1', 'CAR2', 'CAR3']
cell_type_order = ['Neutrophils', 'Macrophages', 'Mast cells','Canonical DCs','Plasmacytoid DCs']
tmp = tmp.loc[sample_order, cell_type_order]

# Create the horizontal stacked bar plot
ax = tmp.plot(kind='bar',stacked=True, edgecolor='none', color=umap_colors)

# Define labels and legend
horiz_offset = 1.03
vert_offset = 1.
ax.legend(bbox_to_anchor=(horiz_offset, vert_offset))
ax.set_ylabel("Normalized Counts")
ax.set_xlabel("Sample")
ax.set_title("Normalized Stacked Bar Plot of Annotation Categories")

# Save the plot
plt.savefig('scRNA_out/figure/myeloid_categories_barplot.pdf', bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Create the cross-tabulated data
tmp = pd.crosstab(subset_adata_my_cmp.obs['label'], subset_adata_my_cmp.obs['cell_type'], normalize='index')

# Define the order of samples and cell types
sample_order = ['UT', 'CAR']
cell_type_order = ['Neutrophils', 'Macrophages', 'Mast cells','Canonical DCs','Plasmacytoid DCs']
tmp = tmp.loc[sample_order, cell_type_order]

# Create the horizontal stacked bar plot
ax = tmp.plot(kind='bar',stacked=True, edgecolor='none', color=umap_colors)

# Define labels and legend
horiz_offset = 1.03
vert_offset = 1.
ax.legend(bbox_to_anchor=(horiz_offset, vert_offset))
ax.set_ylabel("Normalized Counts")
ax.set_xlabel("Sample")
ax.set_title("Normalized Stacked Bar Plot of Annotation Categories")

# Save the plot
plt.savefig('scRNA_out/figure/myeloid_categories_barplot_design.pdf', bbox_inches='tight')

# Show the plot
plt.show()